In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

from pathlib import Path

import numpy as np
import pandas as pd
import os

from code_libraries.acyl_chain_preprocessing import chain_height_df
from code_libraries.bilayer_operations import Bilayer
from code_libraries.op_by_chain import convolve_ops, split_ops_by_label
from code_libraries.predict_acyl_chains_NN_by_chain import clf_lo_ld_by_chain

start_frame, stop_frame = 0, 4000       # same range as the classification
sys_type = "sym"
window = 7                                  # window used in LR_save_classifier_data
red_type = "ft_last"
model_type = "log_reg"
validation_frames = 3

data_array_dir =  os.path.join("../" , "heavy_data_NO_GIT/data_arrays_by_chain_logreg")

In [ ]:
systems = [
    ("dopc_94", 0.0625),
    ("dopc_90", 0.10),
    ("dopc_85", 0.15),
    ("dopc_80", 0.20),
    ("dopc_75", 0.25),
    ("dopc_70", 0.30),
    ("dopc_65", 0.35),
]

chains = ["sn1", "sn2"]
leaflets = ["lower", "upper"]
feature_columns = ["z_height", "xy_radius"]


# Constants mirror chain_height_df(); validation below checks they still match.
sn1_indices = [41, 42, 43, *range(91, 138)]
sn2_indices = [32, 33, 34, *range(44, 91)]
chain_masses = np.array(
    [
        12., 1., 1., 12., 1., 1., 12., 1., 1., 12., 1., 1., 12.,
        1., 1., 12., 1., 1., 12., 1., 1., 12., 1., 12., 1., 12.,
        1., 1., 12., 1., 1., 12., 1., 1., 12., 1., 1., 12., 1.,
        1., 12., 1., 1., 12., 1., 1., 12., 1., 1., 1.,
    ],
    dtype=float,
)


def fast_chain_height_array(dopc_leaflet, chain):
    chain_indices = sn1_indices if chain == "sn1" else sn2_indices
    chain_positions = dopc_leaflet[:, :, chain_indices, :]

    chain_com = np.sum(chain_positions * chain_masses[None, None, :, None], axis=2) / chain_masses.sum()
    shifted_chain_positions = chain_positions - chain_com[:, :, None, :]

    cstart = shifted_chain_positions[:, :, 0, :]
    cend = shifted_chain_positions[:, :, -4, :]

    z_height = np.abs(cstart[:, :, 2] - cend[:, :, 2])
    xy_radius = np.linalg.norm(cstart[:, :, :2] - cend[:, :, :2], axis=2)

    return np.stack([z_height, xy_radius], axis=2)


def mean_pm_std(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return "nan ± nan"
    return f"{values.mean():.4f} ± {values.std(ddof=0):.4f}"


# ---------------------------------------------------------------------------
# Main analysis
# ---------------------------------------------------------------------------

summary_rows = []

for sys_name, cholesterol_concentration in systems:
    print(f"Loading DOPC positions for {sys_name}")

    dopc_pos_raw = clf_lo_ld_by_chain(
        sys_name,
        sys_type,
        "raw",
        model_type,
        "sn1",
        3,
        start_frame,
        stop_frame,
    ).bilayer_all_dopc_data()

    dopc_pos_raw_lower, dopc_pos_raw_upper = dopc_pos_raw
    dopc_by_leaflet = {
        "lower": dopc_pos_raw_lower,
        "upper": dopc_pos_raw_upper,
    }

    chain_class_values = {}

    for chain in chains:
        label_file = os.path.join(data_array_dir, f"{sys_name}_{chain}_{red_type}_{window}pt_prediction_labels.npy")
        #print(f"  Loading saved labels: {label_file.name}")

        preds_lower, preds_upper = np.load(label_file, allow_pickle=True)
        preds_by_leaflet = {
            "lower": np.asarray(preds_lower, dtype=bool),
            "upper": np.asarray(preds_upper, dtype=bool),
        }

        height_lateral_by_leaflet = {}
        labels_by_leaflet = {}

        for leaflet in leaflets:
            dopc_leaflet = dopc_by_leaflet[leaflet]
            n_frames, n_lipids = dopc_leaflet.shape[:2]

            n_check = min(validation_frames, n_frames)
            check_df = chain_height_df(dopc_leaflet[:n_check], n_check, chain, iqr_filter=False)
            # chain_height_df returns rows lipid by lipid (row = lipid * frames + frame) since the
            # acyl_chain_preprocessing update, so the fast (frames, lipids, 2) array is reordered the same way
            check_fast = fast_chain_height_array(dopc_leaflet[:n_check], chain).transpose(1, 0, 2).reshape(-1, 2)

            if list(check_df.columns) != feature_columns:
                raise ValueError(f"{sys_name} {chain} {leaflet}: unexpected feature columns")

            np.testing.assert_allclose(
                check_fast,
                check_df[feature_columns].to_numpy(dtype=float),
                rtol=1e-12,
                atol=1e-12,
            )

            height_lateral = fast_chain_height_array(dopc_leaflet, chain)
            height_lateral_averaged = convolve_ops(height_lateral, window)

            labels = preds_by_leaflet[leaflet]
            if height_lateral_averaged.shape[:2] != labels.shape:
                raise ValueError(
                    f"{sys_name} {chain} {leaflet}: feature shape "
                    f"{height_lateral_averaged.shape[:2]} does not match label shape {labels.shape}"
                )

            height_lateral_by_leaflet[leaflet] = height_lateral_averaged
            labels_by_leaflet[leaflet] = labels

        height_lateral_both = np.concatenate(
            [height_lateral_by_leaflet["lower"], height_lateral_by_leaflet["upper"]],
            axis=1,
        )
        labels_both = np.concatenate(
            [labels_by_leaflet["lower"], labels_by_leaflet["upper"]],
            axis=1,
        )

        lo_features, ld_features = split_ops_by_label(height_lateral_both, labels_both)

        chain_class_values[chain] = {}
        for predicted_class, split_features in [("Ld", ld_features), ("Lo", lo_features)]:
            selected_frames = [frame for frame in split_features if len(frame) > 0]
            selected = np.vstack(selected_frames) if selected_frames else np.empty((0, 2))
            chain_class_values[chain][predicted_class] = selected

    for predicted_class in ["Ld", "Lo"]:
        sn1_values = chain_class_values["sn1"][predicted_class]
        sn2_values = chain_class_values["sn2"][predicted_class]

        summary_rows.append(
            {
                "χc": cholesterol_concentration,
                "predicted_class": predicted_class,
                "sn-1 Height (nm)": mean_pm_std(sn1_values[:, 0]),
                "sn-1 xy_rad (nm)": mean_pm_std(sn1_values[:, 1]),
                "sn-2 Height (nm)": mean_pm_std(sn2_values[:, 0]),
                "sn-2 xy_rad (nm)": mean_pm_std(sn2_values[:, 1]),
            }
        )


summary_df = pd.DataFrame(summary_rows)

output_dir = os.path.join("../", "heavy_data_NO_GIT/results/chain_height_lat_d")
os.makedirs(output_dir, exist_ok=True)
summary_csv = os.path.join(output_dir, f"{model_type}_{window}pt_height_lat_d_summary.csv")
summary_df.to_csv(summary_csv, index=False)

print("\nLR height/xy_rad printed summary table:")
print(summary_df.to_string(index=False))
print(f"\nSaved:\n{summary_csv}")

## Table 2: Lo/Ld percentage differences (height , lateral displacement)

In [ ]:
##Table 2: percentage differences between Lo and Ld chains, (Lo - Ld) / Ld * 100, per concentration,
##then averaged across all concentrations (height and lateral displacement from summary_df above)

def mean_of(entry):
    return float(entry.split("±")[0])

pct_columns = {
    "Height sn-1 (%)": "sn-1 Height (nm)",
    "Lateral Displ. sn-1 (%)": "sn-1 xy_rad (nm)",
    "Height sn-2 (%)": "sn-2 Height (nm)",
    "Lateral Displ. sn-2 (%)": "sn-2 xy_rad (nm)",
}

pct_rows = []
for conc in summary_df["χc"].unique():
    lo_row = summary_df[(summary_df["χc"] == conc) & (summary_df["predicted_class"] == "Lo")].iloc[0]
    ld_row = summary_df[(summary_df["χc"] == conc) & (summary_df["predicted_class"] == "Ld")].iloc[0]
    row = {"χc": conc}
    for pct_name, col in pct_columns.items():
        lo_val, ld_val = mean_of(lo_row[col]), mean_of(ld_row[col])
        row[pct_name] = 100 * (lo_val - ld_val) / ld_val
    pct_rows.append(row)

pct_by_conc = pd.DataFrame(pct_rows)
pct_by_conc.round({name: 2 for name in pct_columns})

In [ ]:
##averaged across all concentrations, formatted as in Table 2 of the manuscript
model_label = {"log_reg": "LR", "xgb": "XGB"}[model_type]      ##name shown in the table

table_2_height_lat = pd.DataFrame(
    [{"ML Model": model_label, **{name: f"{pct_by_conc[name].mean():+.2f}" for name in pct_columns}}]
)
table_2_height_lat

## Table 2: Lo/Ld percentage differences (order parameters , averaged over C2–C17)

In [ ]:
##average order parameter over all 16 carbons (C2–C17) of each chain, split by Lo/Ld,
##percentage difference (Lo - Ld) / Ld * 100 per concentration, then averaged across concentrations

import gc
from code_libraries.acyl_chain_preprocessing import DOPC_op_H_av

##positions within the 46-atom chain lists used by DOPC_op_H_av: (carbon , its hydrogens); C9 and C10 have one H
op_carbon_h = [(0, (1, 2)), (3, (4, 5)), (6, (7, 8)), (9, (10, 11)), (12, (13, 14)), (15, (16, 17)), (18, (19, 20)),
               (21, (22,)), (23, (24,)), (25, (26, 27)), (28, (29, 30)), (31, (32, 33)), (34, (35, 36)),
               (37, (38, 39)), (40, (41, 42)), (43, (44, 45))]
op_sn1_indices = np.array([41, 42, 43, *range(91, 134)])
op_sn2_indices = np.array([32, 33, 34, *range(44, 87)])


def fast_op_array(dopc_leaflet, chain):
    ##vectorized DOPC_op_H_av: (frames , lipids , 138 , 3) -> (frames , lipids , 16); same sign convention
    atoms = dopc_leaflet[:, :, op_sn1_indices if chain == "sn1" else op_sn2_indices, :]
    ops = np.empty(atoms.shape[:2] + (16,))
    for k, (c, hs) in enumerate(op_carbon_h):
        s_k = 0.0
        for h in hs:
            v = atoms[:, :, h, :] - atoms[:, :, c, :]
            cos2 = v[..., 2] ** 2 / np.sum(v ** 2, axis=-1)
            s_k = s_k - (3 * cos2 - 1) / 2
        ops[:, :, k] = s_k / len(hs)
    return ops


op_rows = []

for sys_name, cholesterol_concentration in systems:
    print(f"Order parameters for {sys_name}")

    dopc_pos_raw_lower, dopc_pos_raw_upper = clf_lo_ld_by_chain(
        sys_name, sys_type, "raw", model_type, "sn1", 3, start_frame, stop_frame,
    ).bilayer_all_dopc_data()
    dopc_by_leaflet = {"lower": dopc_pos_raw_lower, "upper": dopc_pos_raw_upper}

    row = {"χc": cholesterol_concentration}

    for chain in chains:
        label_file = os.path.join(data_array_dir, f"{sys_name}_{chain}_{red_type}_{window}pt_prediction_labels.npy")
        preds_lower, preds_upper = np.load(label_file, allow_pickle=True)
        preds_by_leaflet = {"lower": np.asarray(preds_lower, dtype=bool), "upper": np.asarray(preds_upper, dtype=bool)}

        lo_sum, ld_sum, n_lo, n_ld = np.zeros(16), np.zeros(16), 0, 0

        for leaflet in leaflets:
            dopc_leaflet = dopc_by_leaflet[leaflet]

            ##check against DOPC_op_H_av on a few lipids of the first frame
            check_fast = fast_op_array(dopc_leaflet[:1, :3], chain)[0]
            check_ref = np.array([DOPC_op_H_av(dopc_leaflet[0, j], chain) for j in range(3)])
            np.testing.assert_allclose(check_fast, check_ref, rtol=1e-6, atol=1e-6)

            ops_averaged = convolve_ops(fast_op_array(dopc_leaflet, chain), window)     ##(n_red , lipids , 16)
            labels = preds_by_leaflet[leaflet]
            if ops_averaged.shape[:2] != labels.shape:
                raise ValueError(f"{sys_name} {chain} {leaflet}: ops {ops_averaged.shape[:2]} vs labels {labels.shape}")

            ##pool over all lipids and reduced timesteps of both leaflets (as in mean_op_profile)
            lo_sum += ops_averaged[labels].sum(axis=0)
            ld_sum += ops_averaged[~labels].sum(axis=0)
            n_lo += labels.sum()
            n_ld += (~labels).sum()
            del ops_averaged

        lo_profile, ld_profile = lo_sum / n_lo, ld_sum / n_ld          ##(16,) each , C2–C17
        lo_av, ld_av = lo_profile.mean(), ld_profile.mean()           ##average over all carbons
        label = "sn-1" if chain == "sn1" else "sn-2"
        row[f"S_CH Lo {label}"] = lo_av
        row[f"S_CH Ld {label}"] = ld_av
        row[f"S_CH {label} (%)"] = 100 * (lo_av - ld_av) / ld_av

    op_rows.append(row)
    ##free this system's positions before loading the next one
    del dopc_pos_raw_lower, dopc_pos_raw_upper, dopc_by_leaflet, dopc_leaflet
    gc.collect()

op_pct_by_conc = pd.DataFrame(op_rows)
op_pct_by_conc.round(4)

In [ ]:
##averaged across all concentrations; full Table 2 row for this model
table_2 = table_2_height_lat.copy()
table_2["S_CH sn-1 (%)"] = f"{op_pct_by_conc['S_CH sn-1 (%)'].mean():+.2f}"
table_2["S_CH sn-2 (%)"] = f"{op_pct_by_conc['S_CH sn-2 (%)'].mean():+.2f}"

##saved next to the height/lat_d summary (output_dir from the summary cell)
pct_by_conc.merge(op_pct_by_conc, on="χc").to_csv(
    os.path.join(output_dir, f"{model_type}_{window}pt_table2_by_conc.csv"), index=False)
table_2.to_csv(os.path.join(output_dir, f"{model_type}_{window}pt_table2_row.csv"), index=False)
table_2